# Data Preparation for E2SFCA Analysis
## Emergency Obstetric Care Accessibility - Kano State, Nigeria

> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point
from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [6]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Data/raw/'
data_temp = '../Data/processed/'
model_outputs = '../Data/outputs/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [9]:
# Load the healthcare facilities data
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities.geojson')

# Display basic info
print(f"\n✓ Loaded {len(healthcare_facilities_validated)} healthcare facilities")
print(f"CRS: {healthcare_facilities_validated.crs}")

healthcare_facilities_validated.head()


✓ Loaded 145 healthcare facilities
CRS: EPSG:4326


,orig_order,state,lga,ward,urban_conurb,uid,facility_code,ontime_code,facility_name,reg_number,...,longitude,operation_status,registration_status,license_status,created,last_updated,last_updated_ontime,Local_Validation,hcf_id,geometry
0,1210,9,Fagge,Kwachiri,9,12757068.0,19/12/1/2/1/0004,100904010,465 Nigerian Airforce Base Hospital,NaN,...,8.531780,Operational,Registered,Licensed,2018-01-01 01:01:01,2019-12-30 22:56:13,28/09/2022 09:00,Public Comprehensive EmOC,25,POINT (8.53178 12.04531)
1,1208,9,Fagge,Fagge D 2,9,23158449.0,19/12/1/1/1/0001,100904008,Abubakar Imam Urology Centre,NaN,...,8.525738,Operational,Registered,Licensed,2018-01-01 01:01:01,2019-12-30 22:58:50,28/09/2022 09:00,No EmOC,23,POINT (8.52574 12.01474)
2,1302,9,Tarauni,Gyadi-Gyadi Arewa,9,40297833.0,19/21/1/1/2/0004,100911002,Access Clinic,BN 0013966,...,8.541140,Operational,Registered,Licensed,2018-01-01 01:01:01,2020-01-10 20:54:32,28/09/2022 09:00,Private Comprehensive EmOC,114,POINT (8.54114 11.97802)
3,1277,9,Nasarawa,Tudun Wada (NSR),9,42838223.0,19/31/1/2/2/0001,100910010,Ahmadiyya Muslim Hospital,NaN,...,8.548216,Operational,Registered,Licensed,2018-01-01 01:01:01,2020-01-06 14:14:49,28/09/2022 09:00,Private Comprehensive EmOC,92,POINT (8.54822 12.00668)
4,1327,9,Tarauni,Babban Giji,9,NaN,NaN,100911027,Ahmed Memorial Clinic and Maternity,NaN,...,8.535700,Operational,NaN,NaN,NaT,NaT,28/09/2022 09:00,Private Comprehensive EmOC,139,POINT (8.5357 11.96691)


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [ ]:
# Load the study area grid
study_area = gpd.read_file(data_inputs + '100mGrid.gpkg')

# Load the population raster data for Nigeria
raster_path = data_inputs + 'nga_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [ ]:
# Clip the population raster to the study area of Kano State
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

# Update the metadata for the clipped raster
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

# Save the clipped raster to a new file
with rasterio.open(data_inputs + 'kano_nga_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_96338/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


Calculating the centroids for grid cells

In [ ]:
# Extract the population values and their corresponding coordinates from the clipped raster
rows, cols = np.where(band1 > 0)
grid_cells = [clipped_transform * (col + 0.5, row + 0.5) for row, col in zip(rows, cols)]
population_values = band1[rows, cols]

# Create a GeoDataFrame for the population centroids
grid_df = pd.DataFrame(grid_cells, columns=["longitude", "latitude"])
grid_df["population"] = population_values
grid_df["rowid"] = range(1, len(grid_df) + 1)
population_centroids_gdf = gpd.GeoDataFrame(grid_df, geometry=[Point(xy) for xy in zip(grid_df["longitude"], grid_df["latitude"])])
population_centroids_gdf.set_crs("EPSG:4326", inplace=True)

# Save the population centroids GeoDataFrame to a file
population_centroids_gdf.to_file(data_temp + "population_centroids.gpkg", driver="GPKG")
population_centroids_gdf

### Adding population data at 1km grid to 100m grid

In [12]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32632'

In [ ]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-kano.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
print(grid.head())

,grid_id,geometry,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((423886.661 1340202.008, 423996.758 1...",1,12.122137,12.121729,12.122545,8.301005,8.300491,8.301519
1,1,"POLYGON ((425860.71 1334694.134, 425970.815 13...",2,12.072376,12.071968,12.072784,8.319272,8.318758,8.319786
2,2,"POLYGON ((427052.411 1338931.071, 427162.509 1...",3,12.110716,12.110308,12.111124,8.330126,8.329612,8.330640
3,3,"POLYGON ((427046.616 1338660.447, 427156.715 1...",4,12.108269,12.107861,12.108676,8.330079,8.329565,8.330593
4,4,"POLYGON ((427296.472 1329729.352, 427406.583 1...",5,12.027513,12.027105,12.027921,8.332575,8.332061,8.333088
...,...,...,...,...,...,...,...,...,...
167255,167255,"POLYGON ((475971.469 1330011.472, 476081.573 1...",167256,12.030775,12.030368,12.031183,8.779742,8.779228,8.780256
167256,167256,"POLYGON ((475969.606 1329921.277, 476079.71 13...",167257,12.029960,12.029552,12.030368,8.779726,8.779212,8.780240
167257,167257,"POLYGON ((475967.743 1329831.082, 476077.848 1...",167258,12.029144,12.028736,12.029552,8.779709,8.779195,8.780223
167258,167258,"POLYGON ((475965.88 1329740.888, 476075.985 13...",167259,12.028328,12.027921,12.028736,8.779693,8.779179,8.780207


## Building Footprint Data
Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [ ]:
from pathlib import Path
import geopandas as gpd
import duckdb
import pyarrow.parquet as pq
import pyarrow as pa

In [ ]:
# Constants
RELEASE = "2026-01-21.0/"  # oventure data release date

data_inputs = Path("../Kano-scripts/Kano/data-inputs/").resolve()
data_inputs.mkdir(parents=True, exist_ok=True)

boundary_file = data_inputs / "grid-boundary-kano.gpkg"

out_bbox_parquet = data_inputs / "Building-footprint-Kano.parquet"
out_clip_parquet = data_inputs / "Building-footprint-Kano-clipped.parquet"
out_clip_geojson = data_inputs / "Building-footprint-Kano-clipped.geojson"

In [ ]:
# 1. Read the boundary file and get the study area's bounding box in EPSG:4326
gdf = gpd.read_file(boundary_file)
study_area = gdf.dissolve().reset_index(drop=True)

study_area_4326 = study_area.to_crs(4326)
minx, miny, maxx, maxy = study_area_4326.total_bounds
print("bbox:", minx, miny, maxx, maxy)

In [ ]:
# 2. Query the building footprints from Overture Maps using DuckDB and save the results as a Parquet file
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

src = f"s3://overturemaps-us-west-2/release/2026-01-21.0/theme=buildings/type=building/*"

query = f"""
SELECT id, geometry, bbox
FROM read_parquet('{src}', filename=true, hive_partitioning=1)
WHERE
  bbox.xmin <= {maxx} AND bbox.xmax >= {minx}
  AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}
"""

tbl = con.execute(query).fetch_arrow_table()
df = tbl.to_pandas()

# Convert the WKB geometry to GeoDataFrame and set the CRS to EPSG:4326
buildings_4326 = gpd.GeoDataFrame(
    df.drop(columns=["bbox"]),
    geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
    crs=4326
)
print("downloaded rows:", len(buildings_4326))

# Save the building footprints as a Parquet file
buildings_4326.to_parquet(out_bbox_parquet)
print("Saved bbox parquet:", out_bbox_parquet)

In [ ]:
# Clip the building footprints to the study area and save the results as both Parquet and GeoJSON files
buildings = buildings_4326.to_crs(study_area.crs)
clipped = gpd.clip(buildings, study_area)

clipped.to_parquet(out_clip_parquet)
clipped.to_crs(4326).to_file(out_clip_geojson, driver="GeoJSON")

print("Saved clipped parquet:", out_clip_parquet)
print("Saved clipped geojson:", out_clip_geojson)

In [ ]:
# Count buildings per grid cell
# Loading Google building footprints
building_file = data_inputs / 'Kano_GOBv3.gpkg'
buildings = gpd.read_file(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

# Joining buildings to grid
grid_buildings = grid.sjoin(buildings.set_geometry('centroid').drop(columns='geometry'), how='inner', predicate='intersects')
grid_buildings = grid_buildings.groupby('grid_id')

# Counting buildings per grid
building_counts = grid_buildings.size().rename('bcount')

# Adding building count to grid cells
grid = grid.merge(building_counts, on='grid_id', how='left')

# Assign building count 0 to cells with no buildings (NaN)
grid['bcount'] = grid['bcount'].fillna(0)
grid.head()

The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [ ]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Loading coarse pop data
pop_file = data_path / 'kano_nga_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Converting the raster grid to vector data
pop_grid = raster2vector(pop_raster, transform, crs) # rember to save this as a geopackage for future use
pop_grid = pop_grid.to_crs(epsg)
# pop_grid.to_file(data_inputs + 'kano_pop_grid_1km.gpkg', driver='GPKG')

pop_grid['pop_grid_id'] = range(len(pop_grid))

grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid[['pop_grid_pop', 'geometry']], how='left', predicate='within')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

In [ ]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

In [ ]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-kano-centroids.gpkg', driver='GPKG')

## Next Steps

Data is now prepared for E2SFCA analysis. Proceed to:

**→ Notebook 02: E2SFCA Analysis** (`E2SFCA_Analysis.ipynb`)

This notebook will:
1. Calculate distance matrices
2. Run E2SFCA analysis for each facility category
3. Calculate accessibility scores
4. Classify deprivation levels

## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
origin_gdf = population_centroids_gdf
origin_name_column = 'grid_code'
destination_gdf = healthcare_facilities_validated.dropna(subset=['geometry'])
destination_name_column = 'facility_name'

origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [ ]:
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

body = {'locations': locations,
       'destinations': destinations_index,
       'sources': origins_index,
       'metrics': ['distance', 'duration']}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

response = requests.post('https://api.openrouteservice.org/v2/matrix/driving-car', json=body, headers=headers)

In [ ]:
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
# Convert the DataFrame to a GeoDataFrame and save as GeoPackage
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [63]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-kano-centroids.gpkg')
centroids_df

,rowid,latitude,longitude,lon_min,lat_min,lon_max,lat_max,grid_id,bcount,pop_grid_id,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,1,12.122137,8.301005,8.300491,12.121729,8.301519,12.122545,0,10.0,870,110.0,0.090909,120.140793,10.921890,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ..."
1,2,12.072376,8.319272,8.318758,12.071968,8.319786,12.072784,1,1.0,1221,8.0,0.125000,94.052826,11.756603,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ..."
2,3,12.110716,8.330126,8.329612,12.110308,8.330640,12.111124,2,37.0,932,1030.0,0.035922,226.764771,8.145919,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ..."
3,4,12.108269,8.330079,8.329565,12.107861,8.330593,12.108676,3,77.0,932,1030.0,0.074757,226.764771,16.952318,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ..."
4,5,12.027513,8.332575,8.332061,12.027105,8.333088,12.027921,4,41.0,1512,658.0,0.062310,102.597412,6.392848,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167255,167256,12.030775,8.779742,8.779228,12.030368,8.780256,12.031183,167255,0.0,1565,1.0,0.000000,158.605713,0.000000,"POLYGON ((8.77924 12.03118, 8.78026 12.03118, ..."
167256,167257,12.029960,8.779726,8.779212,12.029552,8.780240,12.030368,167256,0.0,1565,1.0,0.000000,158.605713,0.000000,"POLYGON ((8.77923 12.03037, 8.78024 12.03037, ..."
167257,167258,12.029144,8.779709,8.779195,12.028736,8.780223,12.029552,167257,0.0,1565,1.0,0.000000,158.605713,0.000000,"POLYGON ((8.77921 12.02955, 8.78022 12.02955, ..."
167258,167259,12.028328,8.779693,8.779179,12.027921,8.780207,12.028736,167258,0.0,1565,1.0,0.000000,158.605713,0.000000,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8..."


In [64]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'OD-matrix-kano-access-emoc.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,1,1843.72,30.35
1,1,2,1724.08,27.87
2,1,3,1573.44,25.06
3,1,4,1510.41,24.80
4,1,5,2269.93,28.70
...,...,...,...,...
24252695,145,167256,2431.34,37.43
24252696,145,167257,2438.18,37.49
24252697,145,167258,2444.67,37.54
24252698,145,167259,2454.96,37.63


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [65]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,1,1843.72,30.35
1,1,2,1724.08,27.87
2,1,3,1573.44,25.06
3,1,4,1510.41,24.80
4,1,5,2269.93,28.70
...,...,...,...,...
24252695,145,167256,2431.34,37.43
24252696,145,167257,2438.18,37.49
24252697,145,167258,2444.67,37.54
24252698,145,167259,2454.96,37.63


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [66]:
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['rowid', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='rowid', how='left')
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,rowid,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,1,1,1843.72,30.35,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.0,110.0,120.140793,10.921890,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ..."
1,1,2,1724.08,27.87,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,1.0,8.0,94.052826,11.756603,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ..."
2,1,3,1573.44,25.06,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,37.0,1030.0,226.764771,8.145919,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ..."
3,1,4,1510.41,24.80,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,77.0,1030.0,226.764771,16.952318,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ..."
4,1,5,2269.93,28.70,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,41.0,658.0,102.597412,6.392848,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24249650,145,167256,2431.34,37.43,167256,8.779742,12.030775,8.779228,12.030368,8.780256,12.031183,0.0,1.0,158.605713,0.000000,"POLYGON ((8.77924 12.03118, 8.78026 12.03118, ..."
24249651,145,167257,2438.18,37.49,167257,8.779726,12.029960,8.779212,12.029552,8.780240,12.030368,0.0,1.0,158.605713,0.000000,"POLYGON ((8.77923 12.03037, 8.78024 12.03037, ..."
24249652,145,167258,2444.67,37.54,167258,8.779709,12.029144,8.779195,12.028736,8.780223,12.029552,0.0,1.0,158.605713,0.000000,"POLYGON ((8.77921 12.02955, 8.78022 12.02955, ..."
24249653,145,167259,2454.96,37.63,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,0.0,1.0,158.605713,0.000000,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8..."


In [67]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "rowid": "grid_id",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [68]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','facility_name', 'longitude', 'latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left')

In [69]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

In [70]:
category_counts = healthcare_facilities_validated['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Comprehensive EmOC                                 105
Public Comprehensive EmOC                                   17
No EmOC                                                      9
Public/Private Basic EmOC                                    5
Private Basic EmOC                                           5
Non Functional centre .                                      2
Public Basic EmOC                                            1
Public/Private comprehensive EmOC (missionary Hospital)      1
Name: count, dtype: int64


In [71]:
distances_duration_matrix['Local_Validation'] = distances_duration_matrix['Local_Validation'].replace({
    'Public/Private Basic EmOC': 'Private Basic EmOC',
    'Public/Private comprehensive EmOC (missionary Hospital)': 'Private Comprehensive EmOC'
})

In [72]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

In [73]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

In [74]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [75]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return (df.sort_values(['grid_id', 'duration_seconds'])
              .groupby('grid_id', as_index=False)
              .head(n)
              .reset_index(drop=True))

In [76]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

In [77]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,120.140793,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",374.30,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,120.140793,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",1679.81,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,120.140793,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",1708.62,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,94.052826,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",470.08,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,94.052826,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",1560.17,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501712,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,0.000000,0.0,1.0,158.605713,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8...",2057.22,30.54,26,Overcomer's Clinic,8.531794,12.024758,Private Basic EmOC
501713,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,0.000000,0.0,1.0,158.605713,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8...",2080.30,31.62,6,Hanan International Family Clinic,8.513168,12.015043,Private Basic EmOC
501714,167260,8.779676,12.027513,8.779162,12.027105,8.780190,12.027921,158.605713,1.0,1.0,158.605713,"POLYGON ((8.77918 12.02792, 8.78019 12.02792, ...",1915.48,28.10,38,Most Metro Hospital,8.543025,12.019293,Private Basic EmOC
501715,167260,8.779676,12.027513,8.779162,12.027105,8.780190,12.027921,158.605713,1.0,1.0,158.605713,"POLYGON ((8.77918 12.02792, 8.78019 12.02792, ...",2064.03,30.60,26,Overcomer's Clinic,8.531794,12.024758,Private Basic EmOC


In [78]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [79]:
gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG")

In [80]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [81]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [82]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        1    8.301005   12.122137        8.300491       12.121729   
1        1    8.301005   12.122137        8.300491       12.121729   
2        1    8.301005   12.122137        8.300491       12.121729   
3        2    8.319272   12.072376        8.318758       12.071968   
4        2    8.319272   12.072376        8.318758       12.071968   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        8.301519       12.122545   10.921890    10.0            110.0   
1        8.301519       12.122545   10.921890    10.0            110.0   
2        8.301519       12.122545   10.921890    10.0            110.0   
3        8.319786       12.072784   11.756603     1.0              8.0   
4        8.319786       12.072784   11.756603     1.0              8.0   

   pop_grid_pop                                           geometry  \
0    120.140793  POLYGON ((8.30051 12.12254, 8.30152 12.12254, .

In [83]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [84]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [85]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [86]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [87]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",374.30,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,0.166596,1.819541
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",1679.81,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,0.000000,0.000000
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",1708.62,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,0.000000,0.000000
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",470.08,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,0.059205,0.696052
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",1560.17,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501712,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,0.000000,0.0,1.0,...,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8...",2057.22,30.54,26,Overcomer's Clinic,8.531794,12.024758,Private Basic EmOC,0.000000,0.000000
501713,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,0.000000,0.0,1.0,...,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8...",2080.30,31.62,6,Hanan International Family Clinic,8.513168,12.015043,Private Basic EmOC,0.000000,0.000000
501714,167260,8.779676,12.027513,8.779162,12.027105,8.780190,12.027921,158.605713,1.0,1.0,...,"POLYGON ((8.77918 12.02792, 8.78019 12.02792, ...",1915.48,28.10,38,Most Metro Hospital,8.543025,12.019293,Private Basic EmOC,0.000000,0.000000
501715,167260,8.779676,12.027513,8.779162,12.027105,8.780190,12.027921,158.605713,1.0,1.0,...,"POLYGON ((8.77918 12.02792, 8.78019 12.02792, ...",2064.03,30.60,26,Overcomer's Clinic,8.531794,12.024758,Private Basic EmOC,0.000000,0.000000


In [88]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [89]:
origin_dest_sum

,hcf_id,Pop_W
0,1,15603.539814
1,2,36512.813787
2,3,10659.897476
3,4,16032.091170
4,5,55939.302647
...,...,...
129,141,1666.569144
130,142,2708.502060
131,143,5616.732398
132,144,23939.736976


In [ ]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')
origin_dest_acc.head()

In [92]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [93]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [94]:
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_3065/2370967217.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


0          0.000227
1          0.000094
2          0.000117
3          0.000227
4          0.000094
             ...   
1672385    0.000012
1672386    0.000007
1672387    0.000007
1672388    0.000012
1672389    0.000007
Name: supply_demand_ratio, Length: 1672390, dtype: float64

In [95]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [96]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [ ]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])
origin_dest_acc.head()

In [99]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [100]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [40]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [41]:
# Select columns to keep and reorder them
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'hcf_id', 'facility_name', 'Local_Validation', 'duration_seconds', 'distance_km', 'Accessibility_standard', 'geometry']]

In [42]:
# For each grid cell, keep the row with the minimum duration_seconds (closest healthcare facility)
idx = results_grid.groupby('grid_id')['duration_seconds'].idxmin()
results_grid_dedup = results_grid.loc[idx].reset_index(drop=True)
results_grid_dedup

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,hcf_id,facility_name,Local_Validation,duration_seconds,distance_km,Accessibility_standard,geometry
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ..."
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ..."
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ..."
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ..."
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167234,167256,8.779742,12.030775,8.779228,12.030368,8.780256,12.031183,43,Gezawa General Hospital,Public Comprehensive EmOC,868.34,11.64,0.000004,"POLYGON ((8.77924 12.03118, 8.78026 12.03118, ..."
167235,167257,8.779726,12.029960,8.779212,12.029552,8.780240,12.030368,43,Gezawa General Hospital,Public Comprehensive EmOC,875.18,11.70,0.000004,"POLYGON ((8.77923 12.03037, 8.78024 12.03037, ..."
167236,167258,8.779709,12.029144,8.779195,12.028736,8.780223,12.029552,43,Gezawa General Hospital,Public Comprehensive EmOC,881.67,11.75,0.000003,"POLYGON ((8.77921 12.02955, 8.78022 12.02955, ..."
167237,167259,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,43,Gezawa General Hospital,Public Comprehensive EmOC,891.96,11.84,0.000003,"POLYGON ((8.7792 12.02874, 8.78021 12.02874, 8..."


In [50]:
print(results_grid_dedup['Local_Validation'].value_counts())

Local_Validation
Private Comprehensive EmOC    120825
Public Comprehensive EmOC      36954
Private Basic EmOC              9203
Public Basic EmOC                257
Name: count, dtype: int64


In [15]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid_dedup.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [51]:
results_grid_dedup['result'] = 2  # Initialize all cells to 2 (Low accessibility)
results_grid_dedup.loc[results_grid_dedup['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid_dedup.loc[results_grid_dedup['Accessibility_standard'] > 0.02, 'result'] = 0

In [52]:
category_counts = results_grid_dedup['result'].value_counts()
print(category_counts)

result
2    130081
1     24886
0     12272
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [53]:
results_grid_dedup['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.000001) & (results_grid_dedup['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.003) & (results_grid_dedup['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.019) & (results_grid_dedup['Accessibility_standard'] < 0.03), 'focused'] = 1

In [54]:
category_counts = results_grid_dedup['focused'].value_counts()
print(category_counts)

focused
0    147064
1     20175
Name: count, dtype: int64


In [ ]:
results_grid_dedup = results_grid_dedup.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

results_grid_dedup.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,hcf_id,facility_name,Local_Validation,duration_seconds,distance_km,Accessibility_standard,geometry,result,focused
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",2,1
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",2,0
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ...",0,1
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ...",0,1
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ...",2,0


In [57]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid_dedup.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [64]:
# Summarize the results by duration and distance for each category of accessibility (Low, Medium, High)
results_grid_dedup['duration_minutes'] = results_grid_dedup['duration_seconds'] / 60

summary = results_grid_dedup.groupby(['Local_Validation', 'result']).agg({
    'duration_minutes': 'mean',
    'distance_km': 'mean'
}).round(2)

summary = summary.rename(columns={
    'duration_minutes': 'Avg_Duration_Min',
    'distance_km': 'Avg_Distance_KM'
})

summary.index = summary.index.set_levels(
    summary.index.levels[1].map({0: 'Low', 1: 'Medium', 2: 'High'}),
    level=1
)

print(summary)

                                   Avg_Duration_Min  Avg_Distance_KM
Local_Validation           result                                   
Private Basic EmOC         Low                 1.56             0.78
                           Medium              2.75             1.39
                           High               11.59             9.85
Private Comprehensive EmOC Low                 2.21             1.31
                           Medium              3.96             2.66
                           High               12.85            10.66
Public Basic EmOC          Low                 2.07             1.08
                           Medium              3.49             1.71
Public Comprehensive EmOC  Low                 2.96             2.26
                           Medium              4.66             3.48
                           High               13.62            10.70


In [60]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid_dedup.drop(columns=['grid_id', 'hcf_id', 'facility_name', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)
results_table

,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Local_Validation,duration_seconds,distance_km,Accessibility_standard,result,focused
0,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,Public Comprehensive EmOC,374.30,5.77,0.004124,2,1
1,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,Public Comprehensive EmOC,470.08,5.20,0.001466,2,0
2,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,Public Comprehensive EmOC,104.01,0.48,0.021559,0,1
3,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,Public Comprehensive EmOC,40.99,0.22,0.024239,0,1
4,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,Public Comprehensive EmOC,1518.11,12.65,0.000000,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
167234,8.779742,12.030775,8.779228,12.030368,8.780256,12.031183,Public Comprehensive EmOC,868.34,11.64,0.000004,2,0
167235,8.779726,12.029960,8.779212,12.029552,8.780240,12.030368,Public Comprehensive EmOC,875.18,11.70,0.000004,2,0
167236,8.779709,12.029144,8.779195,12.028736,8.780223,12.029552,Public Comprehensive EmOC,881.67,11.75,0.000003,2,0
167237,8.779693,12.028328,8.779179,12.027921,8.780207,12.028736,Public Comprehensive EmOC,891.96,11.84,0.000003,2,0
